# Customer Churn Prediction with Supervised Classifiers
## End-to-End Supervised Learning Workflow

This notebook provides a complete, reproducible walkthrough of the Telco Customer Churn binary classification project.
It covers data inspection, cleaning, exploratory data analysis, pipeline-based preprocessing without data leakage, baseline comparison, 5-fold cross-validation, hyperparameter tuning, out-of-fold threshold selection, final test evaluation, and model interpretation.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('../src')
from data_preprocessing import load_and_clean_data, get_feature_splits, build_preprocessor, RANDOM_STATE

print(f'Random State: {RANDOM_STATE}')

### 1. Data Inspection & Cleaning
- Trims whitespace in string fields.
- Coerces `TotalCharges` to numeric.
- 11 blank `TotalCharges` values belong to new customers (`tenure = 0`). Domain-based imputation sets them to `0.0`.
- Drops `customerID` from predictive features.

In [ ]:
df = load_and_clean_data('../dataset/Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')
print(f'Churn distribution:\n{df["Churn"].value_counts(normalize=True) * 100}')
df.head()

### 2. Stratified Train/Test Split & Preprocessing Pipelines
- 80% Training set (5,634 rows), 20% Test set (1,409 rows).
- Preprocessing uses `ColumnTransformer` with median imputation + standard scaling for numerical features, and most frequent imputation + one-hot encoding for categorical features.
- All transformations are fitted strictly within training folds to eliminate data leakage.

In [ ]:
X_train, X_test, y_train, y_test, num_features, cat_features = get_feature_splits(df)
print(f'Training shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Numerical features ({len(num_features)}): {num_features}')
print(f'Categorical features ({len(cat_features)}): {cat_features}')

### 3. Model Comparison (5-Fold Stratified Cross-Validation on Training Set)
We evaluate a naive `DummyClassifier`, interpretable `LogisticRegression` (with and without class weighting), `DecisionTreeClassifier`, `RandomForestClassifier`, and boosting models.

In [ ]:
with open('../outputs/metrics/cv_comparison_results.json', 'r') as f:
    cv_data = json.load(f)

df_cv = pd.DataFrame(cv_data).T[['val_roc_auc_mean', 'val_pr_auc_mean', 'val_f1_mean', 'val_recall_mean', 'val_precision_mean', 'val_accuracy_mean']]
df_cv.columns = ['ROC-AUC', 'PR-AUC', 'F1-Score', 'Recall', 'Precision', 'Accuracy']
df_cv.sort_values(by='PR-AUC', ascending=False)

### 4. Hyperparameter Tuning Results
Systematic tuning via `GridSearchCV` on the training set optimizing PR-AUC (`average_precision`).

In [ ]:
with open('../outputs/metrics/tuning_results.json', 'r') as f:
    tuning = json.load(f)
for m, details in tuning.items():
    print(f"{m}: Best CV PR-AUC = {details['best_cv_pr_auc']:.4f}")
    print(f"   Params: {details['best_params']}\n")

### 5. Final Evaluation on Untouched Test Set
Evaluating the tuned Gradient Boosting pipeline at default threshold (0.50) and out-of-fold tuned threshold (0.32).

In [ ]:
with open('../outputs/metrics/test_metrics.json', 'r') as f:
    test_metrics = json.load(f)

print('--- Test Set Performance ---')
print(f"Default Threshold (0.50): Accuracy={test_metrics['default_threshold_0.50']['accuracy']:.4f}, "
      f"Recall={test_metrics['default_threshold_0.50']['recall']:.4f}, Precision={test_metrics['default_threshold_0.50']['precision']:.4f}, "
      f"F1={test_metrics['default_threshold_0.50']['f1']:.4f}")
print(f"Tuned Threshold (0.32):   Accuracy={test_metrics['tuned_threshold']['accuracy']:.4f}, "
      f"Recall={test_metrics['tuned_threshold']['recall']:.4f}, Precision={test_metrics['tuned_threshold']['precision']:.4f}, "
      f"F1={test_metrics['tuned_threshold']['f1']:.4f}")
print(f"ROC-AUC: {test_metrics['default_threshold_0.50']['roc_auc']:.4f}, PR-AUC: {test_metrics['default_threshold_0.50']['pr_auc']:.4f}, Brier: {test_metrics['default_threshold_0.50']['brier_score']:.4f}")

### 6. Permutation Feature Importance
Top predictive features on held-out test data.

In [ ]:
with open('../outputs/metrics/permutation_importance.json', 'r') as f:
    fi_data = json.load(f)
pd.DataFrame(fi_data).T.sort_values(by='mean_importance', ascending=False).head(10)